# Crowd People Counter – System Demo
## Development of the AI System

This notebook covers:
1. Dataset preparation and exploratory data analysis (EDA)
2. Fine-tuning YOLOv8n on COCO128
3. Running inference on crowd images
4. Evaluating model performance

> **Screenshot this entire notebook** for the *Development* and *Output Screenshots* sections of the report.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print('Libraries loaded successfully.')

## 1. Dataset – COCO128

COCO128 is a 128-image subset of Microsoft COCO 2017, distributed by Ultralytics.
It downloads automatically when training starts — no manual setup required.

We first inspect the dataset structure and a sample of its images.

In [ ]:
from ultralytics.utils import DATASETS_DIR
from ultralytics.data.utils import check_det_dataset

# Trigger auto-download of COCO128
dataset_info = check_det_dataset('coco128.yaml')
img_dir = Path(dataset_info['train'])

images = sorted(img_dir.glob('*.jpg'))[:5]
print(f'Dataset path  : {img_dir}')
print(f'Total images  : {len(list(img_dir.glob("*.jpg")))}')
print(f'Sample files  : {[p.name for p in images]}')

In [ ]:
# Show resolution distribution across the dataset
all_images = list(img_dir.glob('*.jpg'))
widths, heights = [], []
for p in all_images:
    img = cv2.imread(str(p))
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution'); axes[0].set_xlabel('Width (px)')
axes[1].hist(heights, bins=15, color='coral', edgecolor='white')
axes[1].set_title('Image Height Distribution'); axes[1].set_xlabel('Height (px)')
plt.suptitle('COCO128 – Resolution EDA', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/report_00_eda.png', dpi=150)
plt.show()
print(f'Mean WxH: {sum(widths)//len(widths)} x {sum(heights)//len(heights)} px')

## 2. Model – YOLOv8n Base Model

We load YOLOv8n pre-trained on COCO 2017. The `person` class (index 0) is already well-learned.
Fine-tuning on COCO128 adapts the model weights for the deployment context.

In [ ]:
model = YOLO('../yolov8n.pt')
print(model.info())

## 3. Fine-tuning on COCO128

COCO128 downloads automatically on first run. Set `TRAIN = True` to run fine-tuning.

In [ ]:
TRAIN = False   # Set to True to run fine-tuning (COCO128 auto-downloads)

if TRAIN:
    results = model.train(
        data='coco128.yaml',
        epochs=20,
        imgsz=640,
        batch=16,
        name='crowd_counter',
        project='../models',
        patience=10,
        save=True,
    )
    print('Training complete. Best weights at: ../models/crowd_counter/weights/best.pt')
    model = YOLO('../models/crowd_counter/weights/best.pt')
else:
    print('Skipping training – using pre-trained COCO weights.')

## 4. Inference – Counting People

In [ ]:
PERSON_CLASS = 0
CONF_THRESHOLD = 0.5

results = model('../outputs/crowd.jpg', conf=CONF_THRESHOLD, classes=[PERSON_CLASS], verbose=False)
r = results[0]

count = len(r.boxes)
print(f'People detected: {count}')
print(f'Confidence scores: {[round(c.item(), 3) for c in r.boxes.conf]}')

In [ ]:
annotated = r.plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 7))
plt.imshow(annotated_rgb)
plt.title(f'People Counter Output – {count} people detected (conf ≥ {CONF_THRESHOLD})')
plt.axis('off')
plt.tight_layout()
plt.savefig('../outputs/report_02_detection.png', dpi=150)
plt.show()
print('Saved: outputs/report_02_detection.png')

## 5. CLI Usage

The system is also available as a command-line tool:

In [ ]:
!python ../crowd_counter.py --source ../outputs/crowd.jpg --save